# Create a Session

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("olist-01-ingestion")
    .config("spark.sql.warehouse.dir", "hdfs:///user/hive/warehouse")
    .enableHiveSupport()
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

RAW    = "hdfs:///data/olist/raw"
BUCKET = "gs://olist-sujinan-2026"
DB     = "olist"

def show(df, n=20):
    return df.limit(n).toPandas()

print("warehouse:", spark.conf.get("spark.sql.warehouse.dir"))
print("extensions:", spark.conf.get("spark.sql.extensions", "none"))

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB}")
print(spark.catalog.listDatabases())

# Load data

In [ ]:
FILES = {
    "customers":   "olist_customers_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "categories":  "product_category_name_translation.csv",
}

MULTILINE = {"reviews"}

dfs = {}
for name, filename in FILES.items():
    reader = (spark.read
              .option("header", True)
              .option("inferSchema", True))
    if name in MULTILINE:
        reader = reader.option("multiLine", True).option("escape", '"')
    dfs[name] = reader.csv(f"{RAW}/{filename}")

In [ ]:
EXPECTED = {
    "customers":    99_441,
    "orders":       99_441,
    "order_items": 112_650,
    "payments":    103_886,
    "reviews":      99_224,
    "products":     32_951,
    "sellers":       3_095,
    "geolocation": 1_000_163,
    "categories":       71,
}

bad = []
for name, df in dfs.items():
    n = df.count()
    status = "OK" if n == EXPECTED[name] else f"MISMATCH — expected {EXPECTED[name]:,}"
    if n != EXPECTED[name]:
        bad.append(name)
    print(f"{name:<14s} rows={n:>9,}  cols={len(df.columns):>2}  {status}")

if bad:
    raise ValueError(f"row count mismatch: {bad}")

# Save as bronze tables

In [ ]:
for name, df in dfs.items():
    (df.write
       .mode("overwrite")
       .format("parquet")
       .saveAsTable(f"{DB}.bronze_{name}"))
    print(f"saved {DB}.bronze_{name}")

show(spark.sql(f"SHOW TABLES IN {DB}"))